In [ ]:
import numpy as np 
import pandas as pd 

import matplotlib.pyplot as plt 
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential , Model 
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Dropout, BatchNormalization, AveragePooling2D, GlobalAveragePooling2D
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.applications import MobileNetV2 , VGG16
from tensorflow.keras.optimizers import Adam

import os
import random

import warnings
warnings.filterwarnings('ignore')

In [ ]:
data_path='/kaggle/input/ct-kidney-dataset-normal-cyst-tumor-and-stone/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone'

# Set seed for reproducibility
seed = 123

# Create the train dataset with a validation split
train_ds = keras.utils.image_dataset_from_directory(
    data_path,
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(150, 150),
    validation_split=0.2,
    subset='training',
    seed=seed
)

# Create the validation (test) dataset
test_ds = keras.utils.image_dataset_from_directory(
    data_path,
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(150, 150),
    validation_split=0.2,
    subset='validation',
    seed=seed  
)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
# load MobilenetV2
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(150, 150, 3))

# Freeze layers 
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(4, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=x)

In [ ]:
model.summary()

In [ ]:
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
# Training MobileNetV2
history_mobilenet = model.fit(train_ds, epochs=40, validation_data=test_ds)

In [ ]:
plt.title('MobileNetV2 Accuracy')

plt.plot(history_mobilenet.history['accuracy'], color='red',label='Train')
plt.plot(history_mobilenet.history['val_accuracy'], color='blue',label='Validation')

plt.legend()

In [ ]:
plt.title('MobilenetV2 Loss')

plt.plot(history_mobilenet.history['loss'], color='red',label='Train')
plt.plot(history_mobilenet.history['val_loss'], color='blue',label='Validation')

plt.legend()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# predictions
y_pred = model.predict(test_ds)
y_pred_classes = y_pred.argmax(axis=1)  

y_true = np.concatenate([labels.numpy() for _, labels in test_ds], axis=0)

# classification report
print(classification_report(y_true, y_pred_classes))

# Confusion Matrix
conf_matrix = confusion_matrix(y_true, y_pred_classes)
print(conf_matrix)

In [ ]:
classes = ["Cyst", "Normal", "Stone", "Tumor"]

# สร้าง Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.show()

In [ ]:
# Evaluate the model on the test dataset
test_loss, test_accuracy = model.evaluate(test_ds)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Get a batch of images and their labels from the test set
imgs, clss = next(iter(test_ds.take(1)))

# Make predictions on the batch
pred_prob = np.squeeze(model.predict(imgs, verbose=0))

# Get the predicted class indices
pred_cls = np.argmax(pred_prob, axis=1)

# Class names
class_names = ["Cyst", "Normal", "Stone", "Tumor"]

# Create subplots for displaying images
fig, ax = plt.subplots(4, 4, figsize=(12, 12))
ax = ax.flatten()

# Loop through each image in the batch and display it with predicted and actual class
for i, img in enumerate(imgs[:16]):
    ax[i].imshow(img.numpy().astype("uint8"))
    ax[i].set_xticks([])
    ax[i].set_yticks([])

    # Set the title with the actual and predicted class names, and the prediction probability
    actual_class = clss[i].numpy()
    ax[i].set_title(f'Actual: {class_names[actual_class]}\n'
                     f'Pred: {class_names[pred_cls[i]]} ({round(np.max(pred_prob[i]) * 100, 2)}%)', fontsize=10)

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()


In [ ]:
from tensorflow.keras.applications import EfficientNetB0

# Load EfficientNetB0
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(150, 150, 3))

# Freeze layers
base_model.trainable = False

# Build the custom model on top of EfficientNetB0
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(4, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=x)


In [ ]:
model.summary()

In [ ]:
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
# Training EfficientNetB0
history_efficientnet = model.fit(train_ds, epochs=40, validation_data=test_ds)

In [ ]:
plt.figure(figsize=(10, 4))

# Accuracy
plt.subplot(1, 2, 1)
plt.title('EfficientNetB0 Accuracy')
plt.plot(history_efficientnet.history['accuracy'], color='red', label='Train')
plt.plot(history_efficientnet.history['val_accuracy'], color='blue', label='Validation')
plt.legend()

# Loss
plt.subplot(1, 2, 2)
plt.title('EfficientNetB0 Loss')
plt.plot(history_efficientnet.history['loss'], color='red', label='Train')
plt.plot(history_efficientnet.history['val_loss'], color='blue', label='Validation')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# Evaluate the EfficientNet model on the test dataset
test_loss, test_accuracy = model.evaluate(test_ds)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Get a batch of images and their labels from the test set
imgs, clss = next(iter(test_ds.take(1)))

# Make predictions on the batch
pred_prob = np.squeeze(model.predict(imgs, verbose=0))

# Get the predicted class indices
pred_cls = np.argmax(pred_prob, axis=1)

# Class names
class_names = ["Cyst", "Normal", "Stone", "Tumor"]

# Create subplots for displaying images
fig, ax = plt.subplots(4, 4, figsize=(12, 12))
ax = ax.flatten()

# Loop through each image in the batch and display it with predicted and actual class
for i, img in enumerate(imgs[:16]):
    ax[i].imshow(img.numpy().astype("uint8"))
    ax[i].set_xticks([])
    ax[i].set_yticks([])

    # Set the title with the actual and predicted class names, and the prediction probability
    actual_class = clss[i].numpy()
    ax[i].set_title(f'Actual: {class_names[actual_class]}\n'
                     f'Pred: {class_names[pred_cls[i]]} ({round(np.max(pred_prob[i]) * 100, 2)}%)', fontsize=10)

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()


In [ ]:
# Plot accuracy comparison
plt.figure(figsize=(12, 5))

# Accuracy
plt.subplot(1, 2, 1)
plt.title("Model Accuracy Comparison")
plt.plot(history_mobilenet.history['val_accuracy'], color='orange', label='MobileNetV2')
plt.plot(history_efficientnet.history['val_accuracy'], color='green', label='EfficientNetB0')
plt.xlabel("Epochs")
plt.ylabel("Validation Accuracy")
plt.legend()

# Loss
plt.subplot(1, 2, 2)
plt.title("Model Loss Comparison")
plt.plot(history_mobilenet.history['val_loss'], color='orange', label='MobileNetV2')
plt.plot(history_efficientnet.history['val_loss'], color='green', label='EfficientNetB0')
plt.xlabel("Epochs")
plt.ylabel("Validation Loss")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
def build_unet(input_shape=(150, 150, 3), num_classes=4):
    inputs = keras.Input(shape=input_shape)
    
    # Encoder block 1
    c1 = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    c1 = BatchNormalization()(c1)
    c1 = MaxPooling2D((2, 2))(c1)
    
    # Encoder block 2
    c2 = Conv2D(64, (3, 3), activation='relu', padding='same')(c1)
    c2 = BatchNormalization()(c2)
    c2 = MaxPooling2D((2, 2))(c2)
    
    # Bridge
    b = Conv2D(128, (3, 3), activation='relu', padding='same')(c2)
    b = BatchNormalization()(b)

    # Global pooling instead of upsampling
    x = GlobalAveragePooling2D()(b)
    x = Dropout(0.5)(x)
    
    # Classification head
    x = Dense(128, activation='relu')(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs)
    return model


In [ ]:
unet_model = build_unet()
unet_model.compile(optimizer=Adam(learning_rate=0.0001),
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])
history_unet = unet_model.fit(train_ds, epochs=40, validation_data=test_ds)

In [ ]:
# Comparison Plot
plt.figure(figsize=(14, 6))

# Accuracy Comparison
plt.subplot(1, 2, 1)
plt.title("Model Accuracy Comparison")
plt.plot(history_mobilenet.history['val_accuracy'], color='orange', label='MobileNetV2')
plt.plot(history_efficientnet.history['val_accuracy'], color='green', label='EfficientNetB0')
plt.plot(history_unet.history['val_accuracy'], color='blue', label='U-Net')
plt.xlabel("Epochs")
plt.ylabel("Validation Accuracy")
plt.legend()

# Loss Comparison
plt.subplot(1, 2, 2)
plt.title("Model Loss Comparison")
plt.plot(history_mobilenet.history['val_loss'], color='orange', label='MobileNetV2')
plt.plot(history_efficientnet.history['val_loss'], color='green', label='EfficientNetB0')
plt.plot(history_unet.history['val_loss'], color='blue', label='U-Net')
plt.xlabel("Epochs")
plt.ylabel("Validation Loss")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Predictions for U-Net
y_pred_unet = unet_model.predict(test_ds)
y_pred_classes_unet = y_pred_unet.argmax(axis=1)

# Extract true labels
y_true = np.concatenate([labels.numpy() for _, labels in test_ds], axis=0)

# Classification Report
print("U-Net Classification Report:")
print(classification_report(y_true, y_pred_classes_unet))

# Confusion Matrix
conf_matrix_unet = confusion_matrix(y_true, y_pred_classes_unet)

# Confusion Matrix Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix_unet, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title("U-Net Confusion Matrix")
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.show()


## 📥 Download Trained Model & Artifacts (Kaggle)

This cell saves the trained kidney stone classification models (`kidney_stone_unet_model.keras`, `kidney_stone_unet_model.h5`, `kidney_stone_efficientnet_model.keras`), class mapping (`kidney_stone_classes.json`), creates a zip bundle, and displays direct download links.

In [ ]:
# ====================================================================
# 📥 SAVE & DOWNLOAD TRAINED ARTIFACTS ON KAGGLE
# ====================================================================
import os
import json
import zipfile
from IPython.display import FileLink, display

saved_files = []

# 1. Save U-Net Model (both modern .keras and .h5 formats)
if 'unet_model' in globals():
    unet_keras = 'kidney_stone_unet_model.keras'
    unet_h5 = 'kidney_stone_unet_model.h5'
    unet_model.save(unet_keras)
    unet_model.save(unet_h5)
    saved_files.extend([unet_keras, unet_h5])
    print(f"✓ Saved U-Net model: {unet_keras} and {unet_h5}")

# 2. Save EfficientNet / CNN model if present in session
if 'model' in globals():
    eff_keras = 'kidney_stone_efficientnet_model.keras'
    eff_h5 = 'kidney_stone_efficientnet_model.h5'
    model.save(eff_keras)
    model.save(eff_h5)
    saved_files.extend([eff_keras, eff_h5])
    print(f"✓ Saved CNN model: {eff_keras} and {eff_h5}")

# 3. Save class names mapping
target_classes = globals().get('classes', globals().get('class_names', ["Cyst", "Normal", "Stone", "Tumor"]))
classes_file = 'kidney_stone_classes.json'
with open(classes_file, 'w') as f:
    json.dump({i: name for i, name in enumerate(target_classes)}, f, indent=2)
saved_files.append(classes_file)
print(f"✓ Saved class mapping: {classes_file}")

# 4. Package all files into a single ZIP archive for 1-click download
zip_filename = 'kidney_stone_artifacts.zip'
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file_name in saved_files:
        if os.path.exists(file_name):
            zipf.write(file_name, os.path.basename(file_name))
print(f"✓ Packaged bundle: {zip_filename}")

# 5. Generate Kaggle Direct Download Links
print("\n" + "="*50)
print("👉 Click the link below to download your complete model bundle:")
print("="*50)
display(FileLink(zip_filename))

print("\nIndividual file downloads:")
for file_name in saved_files:
    if os.path.exists(file_name):
        display(FileLink(file_name))
